# Export testovací sady sjízdnosti (ARBot3)

Stáhne z LabelBoxu **tu pevnou 50snímkovou testovací sadu**, na které vznikla čísla
v `SemanticSegmentation.ipynb` (Model61.1 = 0,9546 per-pixel proti ~0,80 histogramu),
zabalí ji a nabídne ke stažení. Cíl: dostat ground truth **do repa** (`models/testset/`),
aby se kvalita segmentace dala měřit proti pravdě i mimo Colab —
`ARBot.Analyze backproject --truth=models/testset`.

**Proč vlastní notebook a ne cela ve velkém:** tenhle nepotřebuje TensorFlow, takže se ho
netýká, že Colab dnes běží na Kerasu 3 a staré `.h5` váhy se v něm nenačtou.

## Jak použít

1. Ulož si LabelBox API klíč do **Colab Secrets** (ikona klíče v levém panelu) pod jménem
   `LABELBOX_API_KEY` a **zapni „Notebook access"** pro tenhle notebook. Klíč do notebooku
   nepatří (viz `CLAUDE.md`); ověřeno, že stačí oprávnění *Project lead*, ne Admin.
2. *Runtime → Run all.*
3. Rozbal stažený `testset.zip` do `models/testset/` v repu.

**Trvá to minuty** — vybrat konkrétní labely dotazem API neumí, takže se streamuje celý
projekt a naše snímky se z něj filtrují.

⚠️ **Nejde to přes `label_generator()` jako velký notebook.** Ten dotaz dnes server odmítá
(`GraphQL validation error`) — legacy API je vypnuté, změřeno 7. 9. 2026. Tady se proto
používá dnešní cesta `project.export()` → `ExportTask` → `get_buffered_stream()`. Až se bude
opravovat velký notebook, je to tentýž problém a totéž řešení.

In [ ]:
# SDK se ZAMERNE NEPRIPINA na 3.x, ackoli velky notebook na te verzi stoji. Zmereno
# 7. 9. 2026: klic i lb.get_project() projdou, ale project.label_generator() server odmitne
# s "GraphQL validation error" - ten dotaz uz na serveru NEEXISTUJE. Pripnuti klienta proto
# nepomuze; zmenila se serverova strana. Dnesni cesta je project.export() -> ExportTask.
!pip install -q labelbox

In [ ]:
import os
import shutil
from io import BytesIO

import numpy as np
import requests
from PIL import Image

PROJECT_ID = 'ck2iqtrixgi6z08113jpu9imr'   # tentyz projekt jako v SemanticSegmentation.ipynb
ORG_ID = 'ck2iqtrgj3wn50757dxn4430t'       # organizace, ve ktere ten projekt je
ANNOTATION = 'Road'                        # jmeno anotace, ktera znamena sjizdno
OUT = './testset'

# Testovaci sada = PEVNY seznam z cely 13 velkeho notebooku (opsany schvalne, aby tenhle
# notebook nezavisel na jeho spusteni). Nemenit - jinak prestanou byt vysledky srovnatelne
# s cisly, ktera se na te sade namerila.
FN_TEST = [
    'ckhfdydqg00543r634y22f0hv', 'ckb188ocv243v0712r8v2g4oa', 'ckhfc30l0001x3r63bn76f1fb',
    'cl4oj3hq61p2j0724fogn01cb', 'cl4pqnqcm2qlb084r4oumel00', 'ckhfd6i4m003t3r63b792nt6a',
    'ck2jg9rq9jbbi0811ykpc4gn8', 'cl4rdsnj70ho707b39ck76aa4', 'ckax08novnzm80753tkvkhmze',
    'ckb17o2n81tsy0744d0d3fy3m', 'cl4rdogoz1hvz07apdkpec2rt', 'cl4oiz8e81pqy07863az37b7q',
    'ckax083sko55u072120pya7tg', 'ckgkw97ys00163q637uvnddz2', 'ckb16pc3z1np30755b8o3h4xe',
    'ckgkw9t8900173q6397ddx7fl', 'ckalh3fxalq510700irq4z7jc', 'ckhfe1pmi005a3r63wejz2paf',
    'cl4rdn73d1hzo077kc6uk74h1', 'ck2jgamqyjbf30811i52b8gpj', 'cl4n80u7x6id3077efm7e7875',
    'ckax0a9abo6hm0712v23ovks3', 'ckhfbsvi9001c3r63dpmnauxe', 'ckb1716l7004p0729g9c9z4bm',
    'cl4nqncdr133y079k2mq52mr1', 'ckhfdmfgi004k3r634npdjfx7', 'ckgkwodoz001r3q634ccfwgng',
    'ckb163dkx1ih80755cg3iqqem', 'ckhfeq9hh00693r636j9an8yh', 'ckhfbkddn00103r63sjghlz0f',
    'cl4oi79m11lrh0786hlts5niy', 'cl4nqq34se1uy077ea2yb1ibi', 'cl4pqp4s75k1d078p3bi7di90',
    'cl4ojh7zm1r1a077xcl2391im', 'ckhfbndya00143r63zzo2caiz', 'ckhfbvg54001g3r63mnib1vhv',
    'ck2jflec7j89w0811w4msy65a', 'ckhfbd348000o3r63opctckcq', 'ckb17msro1muh0753qm1tpyog',
    'ckgkwemq3001e3q63h8ejv40d', 'cl4oj7apc1qw907602qsz7adp', 'cl4ojbd6w1q9g077xfosv97n3',
    'ck2jgixkhjcc3081170b6i6b3', 'cl4oiuzhk1qlc075j2lpgdxpi', 'cl4nqmuwu13pv079i2z7o5z79',
    'ckb164tdd1aqv0753b4r24ktf', 'ckgkvzpv5000u3q63gommxcyh', 'cl4nqytps17fs079idz1l3c2e',
    'ckhfeu0wx006h3r63s8n2q4vl', 'cl4ogqrmq1b5i078670yogd5l',
]


def LabelBoxApiKey(name='LABELBOX_API_KEY'):
  """Klic z Colab Secrets nebo z promenne prostredi. Do notebooku klic NEPATRI.

  ⚠️ Chyby se tu ZAMERNE NEPOLYKAJI a NENI tu interaktivni dotaz. Prvni verze mela
  'except Exception: pass' a fallback na getpass - a to 7. 9. 2026 stalo hledani:
  tajemstvi se nenaslo, getpass v Colabu vratil dict, ten se poslal LabelBoxu jako
  api_key a prisla odpoved "Invalid API key". Tedy hlaska o SPATNEM KLICI v situaci,
  kdy zadny klic nebyl - presne ten druh tiche zamany, kterou zakazuje CLAUDE.md.
  Ted to spadne s duvodem, proc se klic nenasel.
  """
  duvody = []

  try:
    from google.colab import userdata
    try:
      k = userdata.get(name)
    except Exception as e:
      duvody.append('Colab Secrets: %s: %s' % (type(e).__name__, e))
      k = None
    if isinstance(k, str) and k.strip():
      return k.strip()
    if k is not None:
      duvody.append('Colab Secrets vratily %s, ne text' % type(k).__name__)
  except ImportError:
    duvody.append('nebezi to v Colabu (google.colab neni k dispozici)')

  k = os.environ.get(name)
  if isinstance(k, str) and k.strip():
    return k.strip()
  duvody.append('promenna prostredi %s neni nastavena' % name)

  raise RuntimeError(
      'Klic %s se nepodarilo ziskat:\n  - %s\n\n'
      'V Colabu: ikona klice v levem panelu -> Add new secret, jmeno presne %s, '
      'vlozit hodnotu klice a ZAPNOUT prepinac "Notebook access" pro tenhle notebook '
      '(prave ten prepinac se nejcasteji zapomina).' % (name, '\n  - '.join(duvody), name))


print('snimku v testovaci sade:', len(FN_TEST), '| unikatnich:', len(set(FN_TEST)))

In [ ]:
# DIAGNOSTIKA KLICE. Klic sam se NEVYPISUJE - jen jeho necitlive vlastnosti.
# Ma smysl jen na to, co se pozna BEZ site: dostal se klic ze Secrets vubec do kodu,
# a nelepi se na nem bily znak. Jestli je klic PLATNY, rozhodne teprve pokus o prihlaseni.
#
# ⚠️ Formaty klicu jsou dva a nesmi se to zamenit za chybu:
#   - do ~2022 vydaval LabelBox JWT (3 casti oddelene teckou, ~350-400 znaku), ze ktereho
#     se dala precist organizace i platnost;
#   - dnes vydava NEPRUHLEDNY token (~47 znaku), z nehoz se necte nic. To NENI vada.
import base64, json, time

k = LabelBoxApiKey()
print('klic ze Secrets dorazil, delka:', len(k))
print('bile znaky uvnitr:', any(c.isspace() for c in k), '(ma byt False)')

casti = k.split('.')
if len(casti) != 3:
  print('format: nepruhledny token - z hodnoty se nic vycist neda.')
  print('=> o platnosti rozhodne az prihlaseni v dalsi cele.')
else:
  print('format: JWT - jde precist, komu patri:')
  pad = casti[1] + '=' * (-len(casti[1]) % 4)
  p = json.loads(base64.urlsafe_b64decode(pad))
  for pole in ('organizationId', 'apiKeyId', 'userId', 'iat', 'exp'):
    v = p.get(pole)
    if pole in ('iat', 'exp') and v:
      v = '%s (%s UTC)' % (v, time.strftime('%Y-%m-%d %H:%M', time.gmtime(v)))
    print('  %-16s %s' % (pole, v))
  if p.get('exp', 0) < time.time():
    print('!! Klic VYPRSEL - vydej novy.')
  elif p.get('organizationId') != ORG_ID:
    print('!! Klic je z JINE organizace nez projekt (' + ORG_ID + ') - projekt pod nim nebude videt.')
  else:
    print('=> klic patri do spravne organizace a je platny')

In [ ]:
import labelbox as lb
import re
import time

os.makedirs(OUT + '/img', exist_ok=True)
os.makedirs(OUT + '/gt', exist_ok=True)

client = lb.Client(api_key=LabelBoxApiKey())
project = client.get_project(PROJECT_ID)

PRECHODNE = (429, 500, 502, 503, 504)   # ma smysl zkusit znovu
POKUSY = 5


def Stahni(url, popis):
  """Stahne binarku z LabelBoxu, s opakovanim na prechodne chyby.

  ⚠️ Autorizace se posila JEN na api.labelbox.com (masky). Snimky sedi na PODEPSANE URL
  do cloud storage, kam hlavicka Authorization nepatri - s ni se vraci chybova odpoved,
  ktera pak spadne az v PIL jako "cannot identify image file" (7. 9. 2026). Tedy chyba
  hlasena o dva kroky dal, nez je jeji pricina.

  ⚠️ Opakovani je tu proto, ze endpoint pro masky vraci 504 (gateway timeout) - vykresluje
  je server a u velkych masek to nestiha. Bez opakovani spadl cely beh po 26 snimcich.
  """
  posledni = None
  for pokus in range(POKUSY):
    hlavicky = client.headers if 'api.labelbox.com' in url else None
    try:
      r = requests.get(url, headers=hlavicky, timeout=120)
    except Exception as e:
      posledni = 'vyjimka %s: %s' % (type(e).__name__, e)
      r = None
    if r is not None and r.status_code == 200 and r.content:
      return r.content
    if r is not None:
      posledni = 'HTTP %d, %d B' % (r.status_code, len(r.content or b''))
      if r.status_code not in PRECHODNE:
        break
    if pokus < POKUSY - 1:
      cekani = 2 ** pokus
      print('     %s: %s - zkousim znovu za %d s (%d/%d)'
            % (popis, posledni, cekani, pokus + 1, POKUSY - 1))
      time.sleep(cekani)
  raise RuntimeError('%s: %s\n  url: %s' % (popis, posledni, url[:160]))


def NactiObrazek(url, popis):
  data = Stahni(url, popis)
  try:
    return Image.open(BytesIO(data))
  except Exception:
    raise RuntimeError('%s: odpoved NENI obrazek (%d B).\n'
                       '  prvnich 200 B: %r\n  url: %s' % (popis, len(data), data[:200], url[:160]))


want = set(FN_TEST)

# --- Chyby exportu se sbiraji do souhrnu, ne na obrazovku. Podstatne neni, ze jsou, ale
# KOLIK z nich se tyka NASI sady - takove snimky v exportu nebudou a sada bude nekompletni.
chybneLabely, chybneTypy, chybneSchemaId = set(), {}, set()

task = project.export(params={'data_row_details': True, 'label_details': True})
task.wait_till_done()

if task.has_errors():
  def SberChyb(e):
    j = getattr(e, 'json', None) or {}
    if j.get('label_id'):
      chybneLabely.add(j['label_id'])
    t = j.get('error', '?')
    chybneTypy[t] = chybneTypy.get(t, 0) + 1
    # Id chybejiciho nastroje je jen v textu hlasky - bez nej nejde poznat, CO pripojit.
    chybneSchemaId.update(re.findall(r'Unrecognized schema_id: (\w+)', j.get('message', '')))
  task.get_buffered_stream(stream_type=lb.StreamType.ERRORS).start(stream_handler=SberChyb)
  naseChybne = sorted(chybneLabely & want)
  print('!! export hlasi %d chyb u %d labelu; typy: %s'
        % (sum(chybneTypy.values()), len(chybneLabely), chybneTypy))
  print('   z toho z NASI sady: %d z %d' % (len(naseChybne), len(want)))
  if naseChybne:
    print('   (SchemaInconsistencyException = label ma anotaci nastrojem, ktery dnes NENI')
    print('    v ontologii projektu - viz cely "OPRAVA ONTOLOGIE" niz.)')
    print('   dotcene:', naseChybne[:10], '...' if len(naseChybne) > 10 else '')
  print('   chybejici schema id:', sorted(chybneSchemaId))
  print()

if not task.has_result():
  raise RuntimeError('Export nevratil zadny vysledek - viz chyby vyse.')

found, rows, bezAnotace, prazdne, nedotazene = set(), [], [], [], []
# Zmereno 7. 9. 2026: tech 50 jmen jsou ID LABELU (nasly se v poli label.id a taky se
# objevila v chybovem streamu jako label_id). Zkousi se ale porad vsechna tri pole -
# stalo to jeden beh a bez toho by neuspesny filtr vypadal jako prazdny projekt.
zabralo = {}
prikladyId = []


def Zapis(jmeno, img, gt):
  """Ulozi par a vrati radek manifestu. Uklada se AZ kdyz je hotove obe casti."""
  img.save(OUT + '/img/' + jmeno + '.jpg', 'JPEG')
  Image.fromarray(gt).save(OUT + '/gt/' + jmeno + '.png', 'PNG')
  return '%s,%d,%d,%.4f' % (jmeno, img.size[0], img.size[1], float((gt > 0).mean()))


for output in task.get_buffered_stream():
  j = output.json
  dr = j.get('data_row') or {}
  labels = (j.get('projects') or {}).get(PROJECT_ID, {}).get('labels') or []

  kandidati = {'data_row.id': dr.get('id'), 'data_row.external_id': dr.get('external_id')}
  for lab in labels:
    kandidati['label.id'] = lab.get('id')

  jmeno, poleId = None, None
  for pole, v in kandidati.items():
    if isinstance(v, str) and os.path.splitext(v)[0] in want:
      jmeno, poleId = os.path.splitext(v)[0], pole
      break

  if jmeno is None:
    if len(prikladyId) < 3:
      prikladyId.append(dict(kandidati))
    continue
  if jmeno in found:
    continue

  # Uz stazene se preskoci - export trva minuty a masky obcas timeoutuji, takze druhy beh
  # ma dojit dal, ne zacinat znovu. Radek manifestu se dopocita ze souboru na disku.
  cestaImg, cestaGt = OUT + '/img/' + jmeno + '.jpg', OUT + '/gt/' + jmeno + '.png'
  if os.path.exists(cestaImg) and os.path.exists(cestaGt):
    hotovyGt = np.asarray(Image.open(cestaGt))
    hotovyImg = Image.open(cestaImg)
    rows.append('%s,%d,%d,%.4f' % (jmeno, hotovyImg.size[0], hotovyImg.size[1],
                                   float((hotovyGt > 0).mean())))
    found.add(jmeno)
    zabralo[poleId] = zabralo.get(poleId, 0) + 1
    print('%2d/%d  %s  (uz bylo stazene)' % (len(found), len(want), jmeno))
    continue

  try:
    # Obrazek: JPEG s VYCHOZI kvalitou PIL, tedy stejne jako ds_train ve velkem notebooku -
    # jinak by se merilo na jinych datech, nez na kterych vznikla puvodni cisla.
    img = NactiObrazek(dr['row_data'], 'snimek ' + jmeno).convert('RGB')

    # Maska. Dva mozne tvary: per-instance 'mask' (bila plocha na pruhledu) nebo
    # 'composite_mask' celeho labelu, kde se trida pozna PODLE BARVY. Osetruji se oba,
    # aby zmena tvaru neskoncila tichou prazdnou maskou.
    maska, barva = None, (255, 255, 255, 255)
    for lab in labels:
      for a in (lab.get('annotations') or {}).get('objects') or []:
        if a.get('name') != ANNOTATION:
          continue
        perInstance = (a.get('mask') or {}).get('url')
        url = perInstance or (a.get('composite_mask') or {}).get('url')
        if not url:
          continue
        rgb = (a.get('composite_mask') or {}).get('color_rgb')
        if rgb and not perInstance:
          barva = tuple(rgb) + (255,)
        m = np.asarray(NactiObrazek(url, 'maska ' + jmeno).convert('RGBA'), dtype=np.uint8)
        if maska is None:
          maska = m.copy()
        else:
          # Doplnuje se jen tam, kde jeste nic nesviti (alfa 0) - z DownloadLabelBox2.
          dira = maska[:, :, 3] == 0
          maska[dira, :] = m[dira, :]
  except RuntimeError as e:
    # ⚠️ Snimek se NEULOZI ani zpola: nedotazena maska by byla TISE SPATNA pravda,
    # a to je horsi nez chybejici snimek. Dohlasi se na konci.
    nedotazene.append(jmeno)
    print('%2d/%d  %s  PRESKOCENO: %s' % (len(found), len(want), jmeno, str(e).split(chr(10))[0]))
    continue

  if maska is None:
    bezAnotace.append(jmeno)
    gt = np.zeros((img.size[1], img.size[0]), dtype=np.uint8)
  else:
    # Test je na PRESNOU barvu, stejne jako LabelIndexImage ve velkem notebooku - volnejsi
    # prah by dal jinou pravdu, nez na ktere se merilo. Ukladame 0/255 (ne 0/1), aby sla
    # maska otevrit a videt; cteni v ARBot.Analyze na to mysli (prah 128).
    gt = np.where((maska == barva).all(axis=2), 255, 0).astype(np.uint8)

  radek = Zapis(jmeno, img, gt)
  podil = float(radek.split(',')[3])
  if maska is not None and podil == 0.0:
    prazdne.append(jmeno)
  rows.append(radek)
  found.add(jmeno)
  zabralo[poleId] = zabralo.get(poleId, 0) + 1
  print('%2d/%d  %s  %dx%d  sjizdno %.1f %%'
        % (len(found), len(want), jmeno, img.size[0], img.size[1], 100 * podil))

with open(OUT + '/manifest.csv', 'w') as f:
  f.write('name,width,height,road_fraction\n')
  f.write('\n'.join(rows) + '\n')

print()
print('hotovo: %d z %d snimku' % (len(found), len(want)))
print('podle ceho se snimky nasly:', zabralo if zabralo else '(nic)')
if bezAnotace:
  print('!! BEZ ANOTACE "%s" (maska cela nulova): %s' % (ANNOTATION, bezAnotace))
if prazdne:
  print('!! ANOTACE JE, ale po testu na barvu nezbylo nic (jiny tvar masky?): %s' % prazdne)
if nedotazene:
  print('!! NEDOTAZENO (server po %d pokusech): %d snimku - %s' % (POKUSY - 1, len(nedotazene), nedotazene))
  print('   PUST TUHLE CELU ZNOVU - uz stazene preskoci a dojde dal.')

chybi = sorted(want - found)
if chybi:
  vinouOntologie = sorted(set(chybi) & chybneLabely)
  print()
  print('!! NEUPLNE: chybi %d snimku' % len(chybi))
  if vinouOntologie:
    print('   %d kvuli chybe exportu (ontologie): %s' % (len(vinouOntologie), vinouOntologie[:10]))
  if nedotazene:
    print('   %d kvuli timeoutu serveru (staci pustit znovu)' % len(nedotazene))
  jinak = sorted(set(chybi) - chybneLabely - set(nedotazene))
  if jinak:
    print('   %d z JINEHO duvodu (nenaslo se v exportu vubec): %s' % (len(jinak), jinak[:10]))
  if not found:
    print('   Ukazky identifikatoru z exportu - podle nich se pozna, ktere pole FN_TEST odpovida:')
    for k2 in prikladyId:
      print('   ', k2)
  print()
  print('   Sada je pouzitelna i mensi, ale uz NENI presne ta, na ktere vznikla puvodni')
  print('   cisla (0,9546 / ~0,80) - to je potreba pri srovnavani rict.')

In [ ]:
# OPRAVA ONTOLOGIE, KROK 1 - JEN SE CTE, nic se nemeni.
#
# Export selhava na labelech, ktere maji anotaci nastrojem, jenz dnes NENI v ontologii
# projektu. Z chyby je ale jen schema id, kdezto rozhodnout se da jen podle JMENA.
#
# ⚠️ client.get_feature_schema(id) pada na TypeError uvnitr SDK (zmereno 7. 9. 2026).
# To NENI odpoved "neexistuje" - je to pad klienta, a plest si to dvoje znamena vydavat
# vlastni chybu za nalez. Jmena se proto hledaji ve VYPISU knihovne organizace a paruji
# se podle id lokalne; per-id dotaz zustava jen jako zaloha a jeho chyba se hlasi jako pad.
knihovna = {}
chybaVypisu = None
try:
  for fs in client.get_feature_schemas(''):
    n = getattr(fs, 'normalized', None) or {}
    knihovna[fs.uid] = (n.get('name') or getattr(fs, 'name', None) or '(bez jmena)',
                        n.get('tool') or n.get('type') or '?')
except Exception as e:
  chybaVypisu = '%s: %s' % (type(e).__name__, e)


def PopisSchema(schemaId):
  """Vraci (text, dohledano). 'dohledano' znamena, ze JMENO je zname - nic vic."""
  if schemaId in knihovna:
    jmeno, typ = knihovna[schemaId]
    return ('%-22s typ %s' % (jmeno, typ), True)
  try:
    fs = client.get_feature_schema(schemaId)
    n = getattr(fs, 'normalized', None) or {}
    return ('%-22s typ %s' % (n.get('name') or '(bez jmena)',
                              n.get('tool') or n.get('type') or '?'), True)
  except Exception as e:
    return ('jmeno NEZNAME - dotaz na schema spadl (%s: %s). Neznamena to, ze feature '
            'neexistuje.' % (type(e).__name__, e), False)


print('ONTOLOGIE PROJEKTU')
ontology = project.ontology()
print('  id:', ontology.uid, '| jmeno:', ontology.name)
norm = ontology.normalized or {}
print()
print('  nastroje, ktere V NI JSOU (tyhle export uznava):')
for tool in norm.get('tools', []):
  print('     %-22s %-22s %s' % (tool.get('name'), tool.get('tool'), tool.get('featureSchemaId')))
if norm.get('classifications'):
  print('  klasifikace:')
  for c in norm['classifications']:
    print('     %-22s %-22s %s' % (c.get('name'), c.get('type'), c.get('featureSchemaId')))

print()
if chybaVypisu:
  print('!! vypis knihovny feature schemat selhal (%s) - jmena budou nezname' % chybaVypisu)
else:
  print('knihovna organizace ma %d feature schemat' % len(knihovna))

print()
print('NASTROJE, KTERE V ONTOLOGII CHYBI (a proto export tech labelu selhava):')
if not chybneSchemaId:
  print('  zadne - bud export chyby nehlasil, nebo se jeste nepustil.')
popisy = {}
for s in sorted(chybneSchemaId):
  popisy[s] = PopisSchema(s)
  print('  %s' % s)
  print('     %s' % popisy[s][0])

neznama = [s for s in popisy if not popisy[s][1]]
if neznama:
  print()
  print('!! U %d z nich se nepodarilo zjistit jmeno. Pripojit je LZE i tak (jde to podle id),' % len(neznama))
  print('   ale nevis, co pripojujes - a to je spatny zaklad pro schvaleni zmeny.')

In [ ]:
# OPRAVA ONTOLOGIE, KROK 2 - TOHLE UZ MENI PROJEKT V LABELBOXU.
#
# Ve vychozim stavu jen VYPISE, co by udelalo, takze "Run all" nic nezmeni.
# Suchy beh si jmena dohleda SAM - schvalovat zmenu podle neprehlednych id nema smysl.
POTVRZUJI = False

if not chybneSchemaId:
  print('neni co pripojovat.')
else:
  stavajici = [(t.get('name'), t.get('tool')) for t in (ontology.normalized or {}).get('tools', [])]
  seznam = sorted(chybneSchemaId)
  bezJmena = [s for s in seznam if not popisy.get(s, ('', False))[1]]

  print('CO SE ZMENI')
  print('  ontologie: %s (%s)' % (ontology.name, ontology.uid))
  print('  ma dnes %d nastroju:' % len(stavajici))
  for jm, tp in stavajici:
    print('     %-22s %s' % (jm, tp))
  print()
  print('  PRIDA SE do ni %d feature schemat:' % len(seznam))
  for s in seznam:
    print('     %s' % popisy.get(s, ('(nepopsano - pust KROK 1)', False))[0])
    print('        [%s]' % s)
  if bezJmena:
    print()
    print('  !! U %d z nich NENI ZNAME JMENO (dotaz na schema pada uvnitr SDK).' % len(bezJmena))
    print('     Pripojit je lze, ale schvalujes zmenu, o ktere nevis, co pridava.')
  print()
  print('CO SE NEZMENI')
  print('  - anotace v labelech: zustavaji presne jak jsou, nic se nepremaluje ani nemaze')
  print('  - snimky a masky: nedotknute')
  print('  - stavajici nastroje ontologie (vcetne Road): zustavaji')
  print()
  print('DUSLEDKY, se kteryma je potreba pocitat')
  print('  - v editoru LabelBoxu se ty nastroje ZNOVU OBJEVI a bude jimi mozne kreslit')
  print('  - kdyz je ontologie SDILENA s jinymi projekty, projevi se to i tam')
  print('  - vratit zpet = odebrat nastroj z ontologie v LabelBoxu')
  print('  - na zmenu muze byt potreba Admin; klic ma Project lead, pak to skonci chybou')
  print('    o opravneni u konkretniho id, ne tichym nicim')
  print('  - export se timhle NEOPRAVI: musi se pustit ZNOVU exportni cela vyse')
  print()
  print('  A HLAVNE: tohle je potreba jen kvuli tomu, aby export tech labelu vubec prosel.')
  print('  Pravda, kterou merime, je nastroj "Road" - a ten v ontologii JE. Pridavane')
  print('  nastroje na obsah ground truth nemaji vliv.')

  if not POTVRZUJI:
    print()
    print('=== SUCHY BEH, nic se nezmenilo. Kdyz to takhle chces, nastav POTVRZUJI = True')
    print('    a pust tuhle celu znovu.')
  else:
    print()
    print('=== PROVADIM')
    for s in seznam:
      try:
        client.insert_feature_schema_into_ontology(feature_schema_id=s, ontology_id=ontology.uid)
        print('  pripojeno: %s' % s)
      except Exception as e:
        print('  NEPRIPOJENO %s (%s: %s)' % (s, type(e).__name__, e))
    print()
    print('ontologie po zmene:')
    for tool in (client.get_ontology(ontology.uid).normalized or {}).get('tools', []):
      print('  %-22s %-22s %s' % (tool.get('name'), tool.get('tool'), tool.get('featureSchemaId')))
    print()
    print('=> pust ZNOVU exportni celu vyse; teprve ta vyrobi novy vysledek.')

In [ ]:
# Rozliseni se vypise proto, ze se dosud jen predpokladalo (640x480) - a na nem zavisi,
# jak moc squash na 128x128 deformuje pomer stran.
import collections
rozliseni = collections.Counter(r.split(',')[1] + 'x' + r.split(',')[2] for r in rows)
print('rozliseni snimku:', dict(rozliseni))
podily = [float(r.split(',')[3]) for r in rows]
print('podil sjizdne plochy: min %.1f %%  median %.1f %%  max %.1f %%'
      % (100 * min(podily), 100 * sorted(podily)[len(podily) // 2], 100 * max(podily)))

shutil.make_archive('./testset', 'zip', OUT)
print('velikost archivu: %.1f MB' % (os.path.getsize('./testset.zip') / 1024 / 1024))

from google.colab import files
files.download('./testset.zip')